# Matrix Calculus

| Difficulty | ██████░░░░ 6/10 |
| :--- | :--- |
| Prerequisites | 08.01 |
| Time | ~80 min |
| Colab GPU | Not needed |

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NiravRVaghasiya/maths-for-ai/blob/main/08_Advanced_Linear_Algebra/02_matrix_calculus.ipynb)

---

## 🎯 Learning Objective

Learn to differentiate matrix expressions directly using trace tricks, Kronecker products, and the vec operator.

---

## 📐 Theory

[Notebook 02.06](../02_Calculus/06_jacobians_and_hessians.ipynb) built the Jacobian $J$ for
vector-valued functions by literally stacking scalar gradients into a matrix, one row per
output. That works, but for expressions built entirely out of matrix operations — the kind
that fill every neural network layer — recomputing a full Jacobian entry-by-entry is painful
and obscures the structure. **Matrix calculus** gives shortcuts that differentiate a matrix
expression *as a whole*, in one step, and hands back the answer already in matrix form.

### Numerator layout and the gradient of a matrix expression

We adopt **numerator layout**: for scalar $f$ and matrix $X\in\mathbb{R}^{m\times n}$, the
derivative $\partial f/\partial X$ is itself an $m\times n$ matrix with entries
$(\partial f/\partial X)_{ij} = \partial f/\partial X_{ij}$ — same shape as $X$, so it can be
used directly in a gradient-descent update $X \leftarrow X - \eta\,\partial f/\partial X$. Two
building-block rules, both provable by writing out the sum defining each side:

$$\frac{\partial(\mathbf{a}^\top X\mathbf{b})}{\partial X} = \mathbf{a}\mathbf{b}^\top \qquad\qquad
\frac{\partial(\mathbf{a}^\top X^\top X\mathbf{b})}{\partial X} = X(\mathbf{a}\mathbf{b}^\top+\mathbf{b}\mathbf{a}^\top)$$

### The trace trick

Many losses that look like they involve a vector quadratic form are secretly *traces* of a
matrix product, and $\text{tr}(\cdot)$ has clean differentiation rules that a raw sum of
products doesn't show as easily. The key move — the **trace trick** — is rewriting a scalar as
a trace (a scalar is its own $1\times1$ trace) to unlock these identities. A short cheat sheet,
each verified numerically below via finite differences *and* `torch.autograd` before being
trusted:

| Identity | Note |
|---|---|
| $\text{tr}(AB) = \text{tr}(BA)$ | cyclic property — holds whenever both products are defined, even if $A,B$ aren't square |
| $\text{tr}(A) = \text{tr}(A^\top)$ | trace only sees the diagonal, which transpose doesn't move |
| $\dfrac{\partial\,\text{tr}(AX)}{\partial X} = A^\top$ | linear-in-$X$ case; drop the trace and this is just "the gradient of a dot product" |
| $\dfrac{\partial\,\text{tr}(X^\top A X)}{\partial X} = (A+A^\top)X$ | quadratic-in-$X$ case; reduces to $2AX$ when $A$ is symmetric, mirroring $d(x^\top Ax)/dx = 2Ax$ from ordinary calculus |
| $\dfrac{\partial\,\text{tr}(X^\top X)}{\partial X} = 2X$ | special case of the row above with $A=I$: this is exactly $\|X\|_F^2$, the squared Frobenius norm |

The trace trick is what turns "differentiate this loss with respect to a whole weight matrix"
into a two-line derivation instead of tracking $mn$ separate partial derivatives by hand.

### The Kronecker product and the vec operator

The **Kronecker product** $A\otimes B$ (for $A$: $m\times n$, $B$: $p\times q$) is the
$mp\times nq$ block matrix with block $(i,j)$ equal to $A_{ij}B$. The **vec operator**
$\text{vec}(X)$ stacks the columns of $X$ into one long vector. Together they turn matrix
equations into pure linear-algebra ones via the identity

$$\text{vec}(AXB) = (B^\top\otimes A)\,\text{vec}(X)$$

which converts the *matrix-valued linear map* $X\mapsto AXB$ into an ordinary matrix-vector
product acting on $\text{vec}(X)$ — useful whenever you need a Jacobian *of* a matrix expression
*with respect to* a matrix, since "Jacobian of a matrix" isn't directly defined, but "Jacobian
of $\text{vec}$ of a matrix" is just an ordinary Jacobian again, recovering
[02.06](../02_Calculus/06_jacobians_and_hessians.ipynb)'s machinery for a genuinely
matrix-shaped problem.

---

In [ ]:
# Setup
import numpy as np
import torch
import matplotlib.pyplot as plt
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

## 👁️ Visual Intuition

A matrix-calculus identity is really a claim about $mn$ numbers at once — the entire gradient
matrix. Rather than trust the trace-trick formula on faith, we compute the gradient two
completely different ways (finite differences vs. the closed-form identity) and look at every
entry side by side: if the identity is right, the two heatmaps are indistinguishable and their
difference is essentially blank.

In [ ]:
# f(X) = tr(X^T A X) for symmetric A -- claimed gradient (the trace trick's quadratic case)
# is (A + A^T) X = 2AX. Compute the gradient BOTH ways and compare every entry visually.
def numerical_grad(f, X, h=1e-6):
    """Central finite differences, one entry of X at a time -- slow, but assumption-free:
    this doesn't know or care what the closed-form answer is supposed to be."""
    X = X.astype(float)
    grad = np.zeros_like(X)
    for i in range(X.shape[0]):
        for j in range(X.shape[1]):
            X_plus, X_minus = X.copy(), X.copy()
            X_plus[i, j] += h
            X_minus[i, j] -= h
            grad[i, j] = (f(X_plus) - f(X_minus)) / (2 * h)
    return grad

rng = np.random.default_rng(0)
n = 5
A_raw = rng.standard_normal((n, n))
A = A_raw + A_raw.T                       # force symmetric, so (A+A^T)X reduces to 2AX
X0 = rng.standard_normal((n, 3))

f = lambda X: np.trace(X.T @ A @ X)
grad_numeric = numerical_grad(f, X0)
grad_closed_form = (A + A.T) @ X0         # the trace-trick identity's claim

vmax = max(np.abs(grad_numeric).max(), np.abs(grad_closed_form).max())
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, M, title in zip(
    axes, [grad_numeric, grad_closed_form, grad_numeric - grad_closed_form],
    ["Finite-difference gradient\n(no formula assumed)", "Closed-form $(A+A^\\top)X$", "Difference\n(should be ~ blank)"]
):
    im = ax.imshow(M, cmap="coolwarm", vmin=-vmax, vmax=vmax)
    ax.set_title(title, fontsize=10)
    plt.colorbar(im, ax=ax, fraction=0.046)
plt.suptitle("Two independent ways of computing $\\partial\\, \\mathrm{tr}(X^\\top AX)/\\partial X$ agree entry-by-entry")
plt.tight_layout()
plt.show()

print(f"Max abs difference between the two gradient computations: {np.abs(grad_numeric - grad_closed_form).max():.2e}")
print(f"(for reference, the gradient entries themselves are of order {np.abs(grad_closed_form).mean():.2f})")

## 🐍 Implementation from Scratch

For each cheat-sheet identity we check the SAME numeric instance two independent ways: a
from-scratch central-finite-difference gradient (no calculus rules assumed) and
`torch.autograd` (calculus rules assumed, but a completely different code path) — an identity
only earns "verified" once both agree with the closed form. We finish with the
Kronecker/vec identity that turns matrix-shaped derivatives into ordinary ones.

In [ ]:
def finite_diff_grad(f, X, h=1e-6):
    """Central finite differences, entry by entry -- makes zero use of any calculus rule,
    so it's an independent witness for whether a claimed closed-form gradient is correct."""
    X = X.astype(float)
    grad = np.zeros_like(X)
    for i in range(X.shape[0]):
        for j in range(X.shape[1]):
            X_plus, X_minus = X.copy(), X.copy()
            X_plus[i, j] += h
            X_minus[i, j] -= h
            grad[i, j] = (f(X_plus) - f(X_minus)) / (2 * h)
    return grad

rng = np.random.default_rng(1)

# --- Identity 1: tr(AB) = tr(BA), the cyclic property everything else leans on ---
A1, B1 = rng.standard_normal((3, 5)), rng.standard_normal((5, 3))
print("Identity 1  tr(AB) == tr(BA):", np.isclose(np.trace(A1 @ B1), np.trace(B1 @ A1)),
      f"  ({np.trace(A1 @ B1):.4f} vs {np.trace(B1 @ A1):.4f})")

# --- Identity 2: d tr(AX)/dX = A^T ---
m, n = 4, 3
A2, X2 = rng.standard_normal((m, n)), rng.standard_normal((n, m))
grad_fd2 = finite_diff_grad(lambda X: np.trace(A2 @ X), X2)
grad_formula2 = A2.T
X2_t, A2_t = torch.tensor(X2, requires_grad=True), torch.tensor(A2)
torch.trace(A2_t @ X2_t).backward()
print("Identity 2  d tr(AX)/dX = A^T:")
print("  finite-diff vs formula, max abs diff:", np.abs(grad_fd2 - grad_formula2).max())
print("  torch.autograd vs formula, max abs diff:", np.abs(X2_t.grad.numpy() - grad_formula2).max())

# --- Identity 3: d tr(X^T A X)/dX = (A + A^T) X ---
n3 = 4
A3, X3 = rng.standard_normal((n3, n3)), rng.standard_normal((n3, 2))
grad_fd3 = finite_diff_grad(lambda X: np.trace(X.T @ A3 @ X), X3)
grad_formula3 = (A3 + A3.T) @ X3
X3_t, A3_t = torch.tensor(X3, requires_grad=True), torch.tensor(A3)
torch.trace(X3_t.T @ A3_t @ X3_t).backward()
print("\nIdentity 3  d tr(X^T A X)/dX = (A+A^T)X:")
print("  finite-diff vs formula, max abs diff:", np.abs(grad_fd3 - grad_formula3).max())
print("  torch.autograd vs formula, max abs diff:", np.abs(X3_t.grad.numpy() - grad_formula3).max())

# --- Identity 4: d tr(X^T X)/dX = 2X  (A=I special case -- this is d(squared Frobenius norm)) ---
X4 = rng.standard_normal((4, 3))
grad_fd4 = finite_diff_grad(lambda X: np.trace(X.T @ X), X4)
print("\nIdentity 4  d tr(X^T X)/dX = 2X:")
print("  finite-diff vs 2X, max abs diff:", np.abs(grad_fd4 - 2 * X4).max())

# --- Kronecker/vec identity: vec(AXB) = (B^T kron A) vec(X) ---
# order='F' (column-major) matches the standard math-textbook definition of vec(.)
mk, kk, nk, pk = 3, 4, 5, 2
Ak, Xk, Bk = rng.standard_normal((mk, kk)), rng.standard_normal((kk, nk)), rng.standard_normal((nk, pk))
lhs = (Ak @ Xk @ Bk).flatten(order="F")
rhs = np.kron(Bk.T, Ak) @ Xk.flatten(order="F")
print("\nKronecker/vec identity  vec(AXB) = (B^T kron A) vec(X):")
print("  max abs diff:", np.abs(lhs - rhs).max(), f"  (shapes: {Ak.shape} x {Xk.shape} x {Bk.shape})")
print("\nEvery identity holds to within finite-difference/floating-point precision (~1e-9 or better) --")
print("none of these were assumed true, each was checked against an independent computation first.")

## 🤖 Why This Matters for AI

Every dense layer's weight gradient is a matrix-calculus result, not something autograd
invents from nowhere. For $Y = XW^\top + \mathbf{b}$ and squared-error loss
$L=\|Y-Y_{\text{target}}\|_F^2$, the trace trick gives $\partial L/\partial W = (\partial
L/\partial Y)^\top X$ directly — no sum over the batch dimension written by hand, no Python
loop over samples. We derive this exact update below with the trace trick, confirm it against
`torch.autograd`, and confirm the "one matmul instead of a loop" claim is a real speed
difference, not just a stylistic one.

In [ ]:
# Deriving a dense-layer weight gradient with the trace trick, then checking it against
# torch.autograd AND against the naive "loop over every sample" version.
import time

torch.manual_seed(0)
N, d_in, d_out = 256, 128, 64   # batch size, input dim, output dim -- one weight matrix W

X = torch.randn(N, d_in)                          # a batch of inputs
W = torch.randn(d_out, d_in, requires_grad=True)  # the layer's weight matrix
b = torch.randn(d_out, requires_grad=True)
Y_target = torch.randn(N, d_out)

# Forward pass: Y = X W^T + b  (PyTorch's own nn.Linear convention)
Y = X @ W.T + b
loss = ((Y - Y_target) ** 2).sum()
loss.backward()   # autograd's answer -- our ground truth to check against

# Matrix-calculus derivation: L = tr((Y-Yt)^T (Y-Yt)), so dL/dY = 2(Y-Yt) (identity 4's shape,
# generalized off-diagonal). Then Y = XW^T + b is LINEAR in W, so by the chain rule in matrix
# form, dL/dW = (dL/dY)^T @ X -- one matrix multiply, summing over the batch automatically.
dL_dY = 2 * (Y.detach() - Y_target)
dL_dW_matrix_calc = dL_dY.T @ X
dL_db_matrix_calc = dL_dY.sum(dim=0)

print("Matrix-calculus dL/dW matches torch.autograd:", torch.allclose(W.grad, dL_dW_matrix_calc, atol=1e-3))
print("Matrix-calculus dL/db matches torch.autograd:", torch.allclose(b.grad, dL_db_matrix_calc, atol=1e-3))

# Compare against the "obvious" alternative: loop over samples and accumulate outer products.
# Same math (sum of per-sample outer products), but written as Python control flow instead of
# a single matmul -- this is the batch-Jacobian computation matrix calculus lets you skip past.
def gradient_via_python_loop():
    dW = torch.zeros_like(W)
    for i in range(N):
        dW += torch.outer(dL_dY[i], X[i])
    return dW

t0 = time.perf_counter()
dW_loop = gradient_via_python_loop()
t_loop = time.perf_counter() - t0

t0 = time.perf_counter()
for _ in range(20):
    _ = dL_dY.T @ X
t_matmul = (time.perf_counter() - t0) / 20

print(f"\nLoop-based gradient matches the matrix-calculus one: {torch.allclose(dW_loop, dL_dW_matrix_calc, atol=1e-3)}")
print(f"Python loop over {N} samples: {t_loop*1000:.2f} ms")
print(f"Single matmul (dL/dY^T @ X):   {t_matmul*1000:.2f} ms  ({t_loop/t_matmul:.0f}x faster)")
print("\nThe 'batch Jacobian' of a whole layer's weights is, in matrix-calculus form, a single")
print("matmul -- exactly why frameworks like PyTorch vectorize backward passes instead of looping.")

## 🏋️ Exercises

### Warm-up
1. For $A=\begin{pmatrix}1&2\\3&4\end{pmatrix}$ and $X=\begin{pmatrix}0&1\\1&0\end{pmatrix}$,
   compute $\text{tr}(AX)$ and $\text{tr}(XA)$ by hand and confirm they're equal. Then compute
   $\partial\,\text{tr}(AX)/\partial X$ by hand using the cheat sheet and verify with
   `finite_diff_grad`.

### Practice
2. For a NON-symmetric $4\times4$ matrix $A$ (i.e. $A\ne A^\top$), compute
   $\partial\,\text{tr}(X^\top A X)/\partial X$ using the identity $(A+A^\top)X$ and verify it
   against `finite_diff_grad`. Then explain, using the cyclic property $\text{tr}(AB)=\text{tr}(BA)$,
   why $\text{tr}(X^\top A X) = \text{tr}(X^\top A^\top X)$ even when $A \ne A^\top$ (hint: this
   is exactly why only the *symmetric part* $A+A^\top$ of $A$ ever appears in the gradient).

### Challenge
3. Extend the dense-layer gradient derivation to a network with an activation:
   $Y=\sigma(XW^\top+\mathbf{b})$ for $\sigma=\tanh$. Derive $\partial L/\partial W$ by chaining
   the trace-trick result for the linear part with $\sigma'$ applied elementwise (hint: this is
   the Jacobian chain rule from [02.06](../02_Calculus/06_jacobians_and_hessians.ipynb), where
   the elementwise nonlinearity's Jacobian is diagonal). Verify your derivation against
   `torch.autograd` on a random batch.

---

## 📚 Further Reading
- Petersen & Pedersen, [*The Matrix Cookbook*](https://www.math.uwaterloo.ca/~hwolkowi/matrixcookbook.pdf) (the definitive trace-trick/Kronecker reference)
- Magnus & Neudecker, *Matrix Differential Calculus with Applications in Statistics and Econometrics*
- Petersen, Malmgren & Frellsen, ["Auto-Differentiating Linear Algebra"](https://arxiv.org/abs/1710.08717)

---

*Next notebook: [Tensor Decompositions](03_tensor_decompositions.ipynb)*